# 02 — Reading IWFM Input Files (DLL-free)

Every dataset in every IWFM text input file parses into pandas
DataFrames — no raw text is retained, and no DLL is involved. This
notebook tours the readers subsystem by subsystem:

- date utilities and the `24:00` convention
- preprocessor files (nodes, elements, stratigraphy, streams, lakes)
- simulation main
- groundwater package (aquifer parameters, initial heads, subsidence,
  tile drains, element pumping, boundary conditions)
- stream package (reaches, diversions, bypasses)
- root zone (soils, crops, urban, native vegetation)
- small watersheds and the unsaturated zone
- time-series inputs (precip, ET)
- cross-file validation

**Requires:** the sample model (see notebook 01 for how to get it).

In [1]:
import os
from pathlib import Path


def find_sample_model():
    env = os.environ.get("IWFM_SAMPLE_MODEL")
    if env:
        return Path(env)
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / ".assets" / "sample_model"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("sample model not found — see notebook 01")


SAMPLE_MODEL = find_sample_model()
PP_DIR = SAMPLE_MODEL / "Preprocessor"
SIM_DIR = SAMPLE_MODEL / "Simulation"
GW_DIR = SIM_DIR / "GW"
STRM_DIR = SIM_DIR / "Stream"
RZ_DIR = SIM_DIR / "RootZone"

## Dates: the IWFM `24:00` convention

IWFM timestamps look like `MM/DD/YYYY_HH:MM`, and **hour 24:00 means end
of day** — `09/30/1990_24:00` is the instant that October 1 begins.
`parse_iwfm_date` / `format_iwfm_date` handle this convention.

In [2]:
from datetime import datetime

from iwfm_io import format_iwfm_date, parse_iwfm_date

print("parse '09/30/1990_24:00' ->", parse_iwfm_date("09/30/1990_24:00"))
print("format datetime(1990,10,1) ->", format_iwfm_date(datetime(1990, 10, 1)))

parse '09/30/1990_24:00' -> 1990-10-01 00:00:00
format datetime(1990,10,1) -> 09/30/1990_24:00


The parsed instant is correct — `24:00` *is* next-day midnight — but
for grouping, a value stamped at midnight belongs to the day it
**closes**. `iwfm_day` gives that owning day, and `water_year` the
Oct–Sep water year (labeled by ending year); use them instead of
`.dt.year`/`.dt.month` on raw IWFM timestamps.

In [3]:
from iwfm_io import iwfm_day, water_year

print("iwfm_day('09/30/2024_24:00')   ->", iwfm_day("09/30/2024_24:00").date())
print("water_year('09/30/2024_24:00') ->", water_year("09/30/2024_24:00"))
print("water_year('10/01/2024_24:00') ->", water_year("10/01/2024_24:00"))

iwfm_day('09/30/2024_24:00')   -> 2024-09-30
water_year('09/30/2024_24:00') -> 2024
water_year('10/01/2024_24:00') -> 2025


## Preprocessor files

Each file has its own reader returning a dataclass whose `.data` is a
DataFrame. With geopandas installed, spatial tables become GeoDataFrames.

In [4]:
from iwfm_io import read_elements, read_nodes, read_strata

nodes = read_nodes(PP_DIR / "NodeXY.dat")
print(f"{len(nodes.data)} nodes; "
      f"X [{nodes.data['x'].min():.0f}, {nodes.data['x'].max():.0f}], "
      f"Y [{nodes.data['y'].min():.0f}, {nodes.data['y'].max():.0f}]")
nodes.data.head(3)

441 nodes; X [550000, 590000], Y [4400000, 4440000]


,node_id,x,y,geometry
0,1,550000.0,4400000.0,POINT (550000 4400000)
1,2,552000.0,4400000.0,POINT (552000 4400000)
2,3,554000.0,4400000.0,POINT (554000 4400000)


In [5]:
elements = read_elements(PP_DIR / "Element.dat")
print(f"{len(elements.data)} elements in "
      f"{elements.data['subregion'].nunique()} subregions")
elements.data.head(3)

400 elements in 2 subregions


,element_id,node1,node2,node3,node4,subregion
0,1,1,2,23,22,1
1,2,2,3,24,23,1
2,3,3,4,25,24,1


In [6]:
strata = read_strata(PP_DIR / "Strata.dat")
print(f"{len(strata.data)} nodes, {strata.n_layers} aquifer layers")
strata.data.head(3)

441 nodes, 2 aquifer layers


,node_id,elevation,aquitard_1,aquifer_1,aquitard_2,aquifer_2
0,1,500.0,0.0,500.0,10.0,100.0
1,2,500.0,0.0,500.0,10.0,100.0
2,3,500.0,0.0,500.0,10.0,100.0


### The whole tree at once

`read_preprocessor` reads the preprocessor **main** file and follows its
file references automatically. The children are exposed both as a dict
and as direct DataFrame properties.

In [7]:
from iwfm_io import read_preprocessor

pp = read_preprocessor(PP_DIR / "PreProcessor_MAIN.IN")
print("children:", list(pp.children.keys()))
pp.stream_reaches.head(3)

children: ['node', 'element', 'strata', 'stream', 'lake']


,reach_id,n_nodes,outflow_dest,name
0,1,10,-1,Reach1
1,2,6,17,Reach2
2,3,7,0,Reach3


In [8]:
pp.lakes.head(3)

,lake_id,dest_type,dest_id,n_elements,elements
0,1,1,11,10,"[169, 170, 171, 188, 189, 190, 207, 208, 209, ..."


## Simulation main

In [9]:
from iwfm_io import read_simulation

sim = read_simulation(SIM_DIR / "Simulation_MAIN.IN")
print(f"period    {sim.sim_begin} -> {sim.sim_end}")
print(f"time step {sim.time_unit}")
print(f"file refs {len(sim.file_paths)}")

period    09/30/1990_24:00 -> 09/30/2000_24:00
time step 1DAY
file refs 12


## Groundwater package

`read_gw_main` parses the whole GW main file: aquifer parameters (either
a parametric grid when `NGROUP > 0` or per-node values), Kh anomalies,
hydrograph print specs, and initial heads.

In [10]:
from iwfm_io import read_gw_main

gw = read_gw_main(GW_DIR / "GW_MAIN.dat")
print(f"NGROUP = {gw.ngroup} "
      f"({'parametric grid' if gw.ngroup else 'values at every node'})")
print(f"conversion factors: {gw.param_factors}")
print(f"{gw.n_hydrographs} hydrograph print sites")

NGROUP = 1 (parametric grid)
conversion factors: {'fx': 3.2808, 'fkh': 1.0, 'fs': 1e-06, 'fn': 1.0, 'fv': 1.0, 'fl': 1.0}
42 hydrograph print sites


In [11]:
if gw.ngroup:
    # parametric grid: parameter values at parametric nodes, interpolated
    # by IWFM onto the FE grid nodes listed in node_range
    grid = gw.parametric_grids[0]
    print(f"group 1 covers nodes {grid['node_range']} "
          f"({len(grid['nodes'])} nodes)")
    display(grid["params"].head())
else:
    # per-node long format: one row per (node, layer)
    display(gw.aquifer_params.head())

group 1 covers nodes 1-441 (441 nodes)


,node_id,x,y,layer,kh,ss,sy,aquitard_kv,kv
0,1,-15000.0,-15000.0,1,50.0,1.0,0.25,0.2,1.0
1,1,-15000.0,-15000.0,2,50.0,1.0,0.25,0.2,1.0


In [12]:
print(f"initial heads for {len(gw.initial_heads)} nodes (FACTHP={gw.facthp})")
gw.initial_heads.head(3)

initial heads for 441 nodes (FACTHP=1.0)


,node_id,head_layer_1,head_layer_2
0,1,280.0,290.0
1,2,280.0,290.0
2,3,280.0,290.0


In [13]:
# Hydrograph print control — used again in notebook 08 for calibration
gw.hydrographs.head(3)

,id,hydtyp,layer,x,y,node,name,notes
0,1,0,1,574000.0,4440000.0,<NA>,GWHyd1,
1,2,1,1,NaN,NaN,412,GWHyd2,
2,3,1,1,NaN,NaN,391,GWHyd3,


### Subsidence

`sce`/`sci` are elastic/inelastic interbed storage, `dc` interbed
thickness, `hc` pre-compaction head (99999 = use initial heads).

In [14]:
from iwfm_io import read_subsidence

sub = read_subsidence(GW_DIR / "Subsidence.dat")
params = (sub.parametric_grids[0]["params"] if sub.ngroup
          else sub.subsidence_params)
params.head(4)

,node_id,x,y,layer,sce,sci,dc,dcmin,hc
0,1,-15000.0,-15000.0,1,5.0,5.0,10.0,2.0,99999.0
1,1,-15000.0,-15000.0,2,5.0,5.0,10.0,2.0,99999.0


### Tile drains

In [15]:
from iwfm_io import read_tile_drain

td = read_tile_drain(GW_DIR / "TileDrain.dat")
print(f"{td.n_tile_drains} tile drains, {td.n_hydrographs} hydrographs "
      f"-> {td.hyd_out_file}")
td.data.head(3)

21 tile drains, 6 hydrographs -> ..\Results\TileDrainFlows.out


,id,node,elev,conductance,dest_type,dest
0,1,6,280.0,20000.0,1,20
1,2,27,280.0,20000.0,1,20
2,3,48,280.0,20000.0,1,20


### Element pumping

Columns starting with `ic`/`icol` are **pointer columns**: 1-based column
numbers into *other* files (the time-series pumping file, irrigation
fractions, supply adjustment …). Each dataclass docstring says which file
every pointer column references.

In [16]:
from iwfm_io import read_elem_pump

ep = read_elem_pump(GW_DIR / "ElemPump.dat")
print(f"{ep.n_sinks} pumping elements, {ep.n_groups} element groups")
ep.data.head(3)

5 pumping elements, 0 element groups


,id,icolsk,fracsk,ioptsk,fracskl_1,fracskl_2,typdstsk,dstsk,icfirigsk,icadjsk,icskmax,fskmax,name
0,73,1,1.0,0,1.0,1.0,4,1,2,1,0,1.0,
1,193,1,1.0,0,1.0,1.0,4,1,2,1,0,1.0,
2,333,1,1.0,0,1.0,1.0,4,2,1,2,0,1.0,


In [17]:
# element groups in long format (group_id, element_id)
ep.element_groups_df.head()

,group_id,element_id


### Boundary conditions

In [18]:
from iwfm_io import read_bc_main

bc = read_bc_main(GW_DIR / "BC_MAIN.dat")
print(f"{bc.n_bc_hydrographs} boundary-flow hydrographs")

6 boundary-flow hydrographs


## Stream package

In [19]:
from iwfm_io import read_stream_main

sm = read_stream_main(STRM_DIR / "Stream_MAIN.dat")
print(f"{sm.reach_params.shape[0]} reaches, "
      f"{sm.config['n_hydrographs']} hydrograph sites")
sm.reach_params.head(3)

23 reaches, 23 hydrograph sites


,stream_node_id,conductance,bed_thickness,wetted_perimeter,notes
0,1,10.0,1.0,150.0,
1,2,10.0,1.0,150.0,
2,3,10.0,1.0,150.0,


In [20]:
from iwfm_io import read_diver_specs

ds = read_diver_specs(STRM_DIR / "DiverSpecs.dat")
print(f"{ds.n_diversions} diversions; "
      f"delivery groups {ds.delivery_groups_df.shape}, "
      f"recharge zones {ds.recharge_zones_df.shape}")
ds.data[["diversion_id", "export_node", "dest_type", "dest_id",
         "delivery_frac", "name"]]

5 diversions; delivery groups (50, 2), recharge zones (6, 3)


,diversion_id,export_node,dest_type,dest_id,delivery_frac,name
0,1,9,4,2,0.98,UrbanDiversion1
1,2,12,4,2,0.96,UrbanDiversion2
2,3,12,4,1,0.97,AgDiversion1
3,4,22,0,0,0.99,DiverToOutside
4,5,0,6,1,0.99,RiceDiversion


In [21]:
from iwfm_io import read_bypass_specs

bp = read_bypass_specs(STRM_DIR / "BypassSpecs.dat")
n = len(bp.bypass_data) if bp.bypass_data is not None else 0
print(f"{n} bypasses")

2 bypasses


## Root zone

The root-zone main carries the per-element soil table; crop/urban/native
sub-files hang off `rz.file_paths`.

In [22]:
from iwfm_io import read_rootzone_main

rz = read_rootzone_main(RZ_DIR / "RootZone_MAIN.dat")
print(f"{len(rz.element_params)} elements")
rz.element_params.head(3)

400 elements


,element_id,wp,fc,tn,lambda,k,k_ponded,rhc,cap_rise,irne,frne,imsrc,icdstag,icdsturbin,icdsturbout,icdstnvrv
0,1,0.0,0.2,0.45,0.62,2.6,-1.0,2,0.0,1,1.0,0,1,1,1,1
1,2,0.0,0.2,0.45,0.62,2.6,-1.0,2,0.0,1,1.0,0,1,1,1,1
2,3,0.0,0.2,0.45,0.62,2.6,-1.0,2,0.0,1,1.0,0,1,1,1,1


In [23]:
from iwfm_io import read_nonponded_ag_main

np_ag = read_nonponded_ag_main(rz.file_paths["nonponded_ag"])
print(f"{np_ag.n_crops} non-ponded crops: {np_ag.crop_codes}")
np_ag.root_depths

2 non-ponded crops: ['TO', 'AL']


,crop,root_depth,icroot
0,TO,5.0,1
1,AL,6.0,1


In [24]:
from iwfm_io import read_ponded_ag_main, read_urban_main, read_native_veg_main

pa = read_ponded_ag_main(rz.file_paths["ponded_ag"])
ur = read_urban_main(rz.file_paths["urban"])
nv = read_native_veg_main(rz.file_paths["native_veg"])
print(f"ponded root depths:   {pa.root_depths}")
print(f"urban root depth:     {ur.root_depth}")
print(f"native/riparian:      {nv.root_depth_native} / {nv.root_depth_riparian}")
ur.element_params.head(2)

ponded root depths:   {'rice_fl': 2.0, 'rice_nfl': 2.0, 'rice_ndc': 2.0, 'refuge_sl': 2.0, 'refuge_pr': 2.0}
urban root depth:     2.0
native/riparian:      3.0 / 3.0


,element_id,perv_fraction,cn,icpopul,icwtruse,fracdm,iceturb,icrtfurb,icrufurb,icurbspec
0,1,0.5,65.0,1,1,0.0,4,2,2,1
1,2,0.5,65.0,1,1,0.0,4,2,2,1


## Small watersheds and unsaturated zone

In [25]:
from iwfm_io import read_swshed

sw = read_swshed(SIM_DIR / "SWShed.dat")
print(f"{sw.n_watersheds} small watersheds "
      f"(tables: watershed_data, watershed_nodes, rootzone_params, "
      f"aquifer_params, initial_conditions)")
sw.watershed_data

3 small watersheds (tables: watershed_data, watershed_nodes, rootzone_params, aquifer_params, initial_conditions)


,id,area,stream_node,n_gw_nodes
0,1,6.0,1,2
1,2,6.0,3,3
2,3,6.0,21,2


In [26]:
from iwfm_io import read_unsatzone

uz = read_unsatzone(SIM_DIR / "UnsatZone.dat")
print(f"{uz.n_unsat_layers} unsaturated layers, "
      f"{uz.element_params['element_id'].nunique()} elements")
uz.element_params.head(4)

2 unsaturated layers, 400 elements

,element_id,layer,thickness,porosity,pore_size_index,k,rhc
0,1,1,20.0,0.3,0.35,10.0,2
1,1,2,20.0,0.3,0.35,10.0,2
2,2,1,20.0,0.3,0.35,10.0,2
3,2,2,20.0,0.3,0.35,10.0,2


## Time-series inputs

Time-series files either carry inline data (a DataFrame) or point at a
HEC-DSS file (a list of DSS pathnames per column) — the sample model has
one of each.

In [27]:
from iwfm_io import read_et, read_precip

for name, ts in [("Precip", read_precip(SIM_DIR / "Precip.dat")),
                 ("ET", read_et(SIM_DIR / "ET.dat"))]:
    if ts.data is not None:
        print(f"{name}: {len(ts.data)} timesteps x {ts.spec.n_columns} cols "
              f"(inline)")
    else:
        print(f"{name}: {ts.spec.n_columns} cols from DSS file "
              f"{ts.spec.dss_file!r}")
        for col, pathname in ts.dss_pathnames[:2]:
            print(f"   col {col}: {pathname}")

Precip: 2 cols from DSS file 'TSDATA_IN.DSS'
   col 1: /SAMPLE_PROBLEM/GAGE1/PRECIP//1MON/PRECIPITATION/
   col 2: /SAMPLE_PROBLEM/GAGE2/PRECIP//1MON/PRECIPITATION/
ET: 12 timesteps x 7 cols (inline)


In [28]:
read_et(SIM_DIR / "ET.dat").data.head(3)

,date,col_1,col_2,col_3,col_4,col_5,col_6,col_7
0,10/31/4000_24:00,3.4,3.5,2.2,3.4,3.4,3.4,3.7
1,11/30/4000_24:00,1.6,1.6,1.6,1.6,1.6,1.6,1.8
2,12/31/4000_24:00,1.0,1.0,1.0,0.5,1.0,1.2,1.2


## Cross-file validation

`validate_preprocessor` (and the lower-level `validate_nodes` /
`validate_elements` / `validate_stratigraphy`) run consistency checks
across the parsed files — element node references, layer counts, and so
on. An empty list means no problems.

In [29]:
from iwfm_io import validate_preprocessor

errors = validate_preprocessor(pp)
print("validation:", errors if errors else "no errors")

validation: no errors


## Recap

- one `read_*` function per file; dataclasses hold every dataset as a
  DataFrame
- `read_preprocessor` / `read_rootzone_main` follow file references
- pointer columns (`ic*`, `irn*`, `itscol*`) are 1-based column numbers
  into other files — the dataclass docstrings say which
- notebook 04 shows the mirror-image `write_*` functions